<a href="https://colab.research.google.com/github/Rafael-Labarrere/Machine-Learning/blob/Trabalho-Final/trabalho_final_igor_rafael_v03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Trabalho Final - Inteligência Artificial
## 📌 Tema: Modelo para Prever o Salário em vagas de T.I e sua evolução ao longo do tempo

#👾 CÓDIGO:

In [1]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google'

In [2]:
# Instalação das dependências atualizadas com suporte a PyTorch (padrão mais estável para HuggingFace)
%pip install transformers torch geopy country-converter scikit-learn --quiet

import torch
import torch.nn as nn
from transformers import BertTokenizer, BertModel

# Definição das constantes globais para evitar erros de 'not defined' nas células subsequentes
MODEL_NAME = 'bert-base-uncased'
MAX_LEN = 128


Note: you may need to restart the kernel to use updated packages.


C:\Users\Rafael\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Imports utilitários atualizados (sem dependências do TensorFlow/Keras)
from geopy.geocoders import Nominatim
import country_converter as coco
from sklearn.utils import resample


In [5]:
# Estas variáveis DEVEM estar definidas antes:
MODEL_NAME = 'bert-base-uncased'  # ou outro modelo BERT
MAX_LEN = 128                     # ou outro valor adequado
# X_structured_train_processed deve existir do pré-processamento

# Passo 1: Ajustes no Pré-processamento


In [ ]:
# ==============================================================================
# 1. IMPORTS E CONFIGURAÇÕES
# ==============================================================================
import pandas as pd
import numpy as np
import torch
from transformers import BertTokenizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.utils import resample
import country_converter as coco
from geopy.geocoders import Nominatim
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Configurações visuais
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 7)

# ==============================================================================
# 2. CARREGAMENTO DOS DADOS 
# ==============================================================================
csv_path = "data/ai_job_dataset.csv"
df = pd.read_csv(csv_path)
# ==============================================================================
# NOVO CÓDIGO - ENRIQUECIMENTO DE FEATURES GEOGRÁFICAS
# ==============================================================================
def enrich_location_data(df):
    # Padronizar códigos de país
    df['country_code'] = coco.convert(names=df['company_location'], to='ISO3')
    df['continent'] = coco.convert(names=df['company_location'], to='continent')

    # Exemplo simplificado - na prática use dados reais
    gdp_map = {'USA': 1.0, 'GBR': 0.8, 'IND': 0.2}
    df['gdp_weight'] = df['country_code'].map(gdp_map).fillna(0.5)

    return df

df = enrich_location_data(df)

# ==============================================================================
# 3. PRÉ-PROCESSAMENTO E ENGENHARIA DE FEATURES
# ==============================================================================

# 3.1. Combinação de campos textuais e seleção de features
df['text_input'] = df['job_title'] + " " + df['required_skills']

categorical_features = ['experience_level', 'employment_type', 'company_location',
                        'company_size', 'employee_residence', 'education_required',
                        'industry', 'salary_currency', 'company_name', 'job_title']
numerical_features = ['remote_ratio', 'years_experience', 'job_description_length', 'benefits_score']
text_feature = 'text_input'
target = 'salary_usd'

# 3.2. Tratamento de Valores Ausentes
# Para colunas numéricas, preenchemos com a mediana (mais robusta a outliers)
for col in numerical_features:
    median_val = df[col].median()
    df[col].fillna(median_val, inplace=True)

# Para colunas categóricas, preenchemos com a moda (valor mais frequente)
for col in categorical_features:
    mode_val = df[col].mode()[0]
    df[col].fillna(mode_val, inplace=True)

print("\nTratamento de valores ausentes concluído.")
print(f"Valores nulos restantes no dataframe: {df.isnull().sum().sum()}")

# ==============================================================================
# NOVO CÓDIGO - BALANCEAMENTO POR REGIÃO
# ==============================================================================
def balance_by_region(df, region_col='continent'):
    regions = df[region_col].unique()
    min_size = df[region_col].value_counts().min()

    balanced_dfs = []
    for region in regions:
        region_df = df[df[region_col] == region]
        balanced_dfs.append(
            resample(region_df,
                   replace=len(region_df) < min_size,
                   n_samples=min_size,
                   random_state=42)
        )
    return pd.concat(balanced_dfs)

df_balanced = balance_by_region(df)

# 3.3. Divisão dos dados em Treino e Validação
X = df_balanced.drop(target, axis=1)
y = df_balanced[target]
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# 3.4. Processamento das Features Estruturadas (Numéricas e Categóricas)
# Usaremos um ColumnTransformer que aplica os transformadores corretos a cada tipo de coluna.
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor_structured = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='drop'
)

# Ajustamos o preprocessador SOMENTE nos dados de treino
X_structured_train_processed = preprocessor_structured.fit_transform(X_train)
# E aplicamos a transformação nos dados de validação
X_structured_val_processed = preprocessor_structured.transform(X_val)

# 3.5. Tokenização dos Textos com BertTokenizer
MODEL_NAME = 'bert-base-uncased'
MAX_LEN = 128
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

def bert_tokenizer_func(texts, tokenizer, max_len):
    return tokenizer(
        texts.tolist(),
        add_special_tokens=True,
        max_length=max_len,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt' # Retorna tensores do PyTorch (Versão corrigida)
    )

# Tokenizando os textos de treino e validação
train_encodings = bert_tokenizer_func(X_train[text_feature], tokenizer, MAX_LEN)
val_encodings = bert_tokenizer_func(X_val[text_feature], tokenizer, MAX_LEN)

print("\nPré-processamento e tokenização concluídos com PyTorch!")
print(f"Shape dos dados estruturados de treino: {X_structured_train_processed.shape}")
print(f"Shape dos dados de texto (input_ids) de treino: {train_encodings['input_ids'].shape}")


# Passo 2: Correção da Arquitetura do Modelo

In [ ]:
# ==============================================================================
# 4. CRIAÇÃO DO MODELO HÍBRIDO EM PYTORCH (VERSÃO CORRIGIDA E MODERNA)
# ==============================================================================
import torch
import torch.nn as nn
from transformers import BertModel

class HybridBertModel(nn.Module):
    def __init__(self, bert_model_name, num_structured_features):
        super(HybridBertModel, self).__init__()
        # Carrega o BERT nativo em PyTorch
        self.bert = BertModel.from_pretrained(bert_model_name)
        
        # Congela os pesos do BERT para acelerar o treinamento básico (opcional)
        for param in self.bert.parameters():
            param.requires_grad = False
            
        # Dimensão de saída do BERT base é 768
        self.dense = nn.Sequential(
            nn.Linear(768 + num_structured_features, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )

    def forward(self, input_ids, attention_mask, structured_feat):
        # Extração de características de texto via BERT
        bert_outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # Usando o pooler_output (equivalente ao token [CLS] processado)
        text_features = bert_outputs.pooler_output
        
        # Concatenação dos embeddings do texto com os dados estruturados
        combined = torch.cat((text_features, structured_feat), dim=1)
        
        # Regressão linear
        output = self.dense(combined)
        return output

# Criar instância do modelo
try:
    # X_structured_train_processed.shape[1] contém a dimensão dos dados tabulares processados
    num_features = X_structured_train_processed.shape[1]
    hybrid_model = HybridBertModel(MODEL_NAME, num_features)
    print("\nModelo Híbrido baseado em PyTorch criado com sucesso!")
    print(f"Estrutura do Modelo:\n{hybrid_model}")
except Exception as e:
    print(f"Erro ao criar o modelo: {str(e)}")


In [9]:
# ==============================================================================
# 4. CRIAÇÃO E RESUMO DO MODELO HÍBRIDO (VERSÃO CORRIGIDA)
# ==============================================================================

# 1. Verificar shape dos dados estruturados
print(f"Shape dos dados estruturados: {X_structured_train_processed.shape}")

# 2. Definir a arquitetura do modelo
def create_hybrid_bert_model(bert_model_name, max_len, num_structured_features):
    # Camadas de entrada
    input_ids = Input(shape=(max_len,), dtype=tf.int32, name='input_ids')
    attention_mask = Input(shape=(max_len,), dtype=tf.int32, name='attention_mask')
    structured_input = Input(shape=(num_structured_features,), name='structured_input')

    # Camada BERT
    bert_model = TFBertModel.from_pretrained(bert_model_name)
    bert_output = bert_model(input_ids, attention_mask=attention_mask).last_hidden_state

    # Pooling e concatenação
    text_features = GlobalAveragePooling1D()(bert_output)
    combined = Concatenate()([text_features, structured_input])

    # Camadas densas
    x = Dense(128, activation='relu', kernel_regularizer=l1_l2(0.01, 0.01))(combined)
    x = Dropout(0.3)(x)
    output = Dense(1, activation='linear')(x)

    return Model(inputs=[input_ids, attention_mask, structured_input], outputs=output)

# 3. Criar e compilar o modelo
try:
    hybrid_model = create_hybrid_bert_model(MODEL_NAME, MAX_LEN, X_structured_train_processed.shape[1])

    hybrid_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=3e-5),
        loss='mse',
        metrics=['mae', tf.keras.metrics.RootMeanSquaredError(name='rmse')]
    )

    # 4. Mostrar resumo detalhado
    print("\n" + "="*80)
    print("RESUMO DO MODELO HÍBRIDO")
    print("="*80)
    hybrid_model.summary(show_trainable=True, line_length=100)

    # Mostrar informações adicionais
    print("\nInformações Adicionais:")
    print(f"- Total de parâmetros: {hybrid_model.count_params():,}")
    print(f"- Parâmetros treináveis: {sum([tf.keras.backend.count_params(w) for w in hybrid_model.trainable_weights]):,}")
    print(f"- Camadas: {len(hybrid_model.layers)}")

except Exception as e:
    print(f"\nERRO ao criar o modelo: {str(e)}")
    print("Verifique se:")
    print("1. Todas as variáveis necessárias estão definidas")
    print("2. Os imports estão corretos")
    print("3. O pré-processamento foi executado completamente")

Shape dos dados estruturados: (2342, 113)

ERRO ao criar o modelo: name 'Input' is not defined
Verifique se:
1. Todas as variáveis necessárias estão definidas
2. Os imports estão corretos
3. O pré-processamento foi executado completamente


#

# Passo 3: Treinamento do Modelo

In [10]:
# ==============================================================================
# 4. CRIAÇÃO DO MODELO HÍBRIDO EM PYTORCH (VERSÃO CORRIGIDA E MODERNA)
# ==============================================================================
import torch
import torch.nn as nn
from transformers import BertModel

class HybridBertModel(nn.Module):
    def __init__(self, bert_model_name, num_structured_features):
        super(HybridBertModel, self).__init__()
        # Carrega o BERT nativo em PyTorch
        self.bert = BertModel.from_pretrained(bert_model_name)

        # Congela os pesos do BERT para acelerar o treinamento básico (opcional)
        for param in self.bert.parameters():
            param.requires_grad = False

        # Dimensão de saída do BERT base é 768
        self.dense = nn.Sequential(
            nn.Linear(768 + num_structured_features, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )

    def forward(self, input_ids, attention_mask, structured_feat):
        # Extração de características de texto via BERT
        bert_outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # Usando o pooler_output (equivalente ao token [CLS] processado)
        text_features = bert_outputs.pooler_output

        # Concatenação dos embeddings do texto com os dados estruturados
        combined = torch.cat((text_features, structured_feat), dim=1)

        # Regressão linear
        output = self.dense(combined)
        return output

# Criar instância do modelo
try:
    hybrid_model = HybridBertModel(MODEL_NAME, X_structured_train_processed.shape[1])
    print("Modelo Híbrido baseado em PyTorch criado com sucesso!")
except Exception as e:
    print(f"Erro ao criar o modelo: {str(e)}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 674.35it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modelo Híbrido baseado em PyTorch criado com sucesso!


In [ ]:
# ==============================================================================
# 5. TREINAMENTO DE VERIFICAÇÃO EM PYTORCH (50 PASSOS)
# ==============================================================================
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# 1. Configurar dispositivo (GPU se disponível, senão CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
hybrid_model.to(device)

# 2. Converter dados para Tensores do PyTorch
# Os encodings já são tensores de PyTorch devido ao preenchimento anterior
X_train_inputs = {
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask'],
    'structured_input': X_structured_train_processed
}
train_input_ids = X_train_inputs['input_ids']
train_attention_mask = X_train_inputs['attention_mask']
train_structured = torch.tensor(X_train_inputs['structured_input'], dtype=torch.float32)
train_targets = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)

# 3. Criar DataLoader para gerenciar os lotes (Batches)
dataset = TensorDataset(train_input_ids, train_attention_mask, train_structured, train_targets)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

# 4. Definir Otimizador e Função de Perda (MSE)
optimizer = optim.Adam(hybrid_model.parameters(), lr=3e-5)
criterion = torch.nn.MSELoss()

print(f"Iniciando treinamento de verificação no dispositivo: {device}")
hybrid_model.train()

# Executar exatamente 50 passos de treino
steps = 0
running_loss = 0.0
max_steps = 50

for epoch in range(1):
    for batch in dataloader:
        if steps >= max_steps:
            break

        # Descompactar lote e mover para o dispositivo correspondente
        b_input_ids, b_attn_mask, b_structured, b_targets = [t.to(device) for t in batch]

        # Zerar os gradientes do otimizador
        optimizer.zero_grad()

        # Forward pass
        outputs = hybrid_model(b_input_ids, b_attn_mask, b_structured)

        # Calcular a perda
        loss = criterion(outputs, b_targets)

        # Backward pass e otimização
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        steps += 1

        if steps % 10 == 0:
            print(f"Passo [{steps}/{max_steps}] | Perda (MSE): {running_loss / steps:.4f}")

print("\nVerificação concluída com sucesso em PyTorch!")

NameError: name 'X_train_inputs' is not defined

## 2.2 MODELO COMPLETO E MAIS SEGURO:

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint

# ==============================================================================
# 5. TREINAMENTO COMPLETO E SEGURO
# ==============================================================================

# (O código para preparar X_train_inputs e X_val_inputs continua o mesmo)
X_train_inputs = {
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask'],
    'structured_input': X_structured_train_processed
}
X_val_inputs = {
    'input_ids': val_encodings['input_ids'],
    'attention_mask': val_encodings['attention_mask'],
    'structured_input': X_structured_val_processed
}

# Parâmetros do treinamento completo
EPOCHS = 5
BATCH_SIZE = 16

# Callback para salvar o melhor modelo durante o treino
checkpoint_filepath = 'melhor_modelo_pesos.weights.h5'
model_checkpoint_callback = ModelCheckpoint(
    filepath=checkpoint_filepath,
    save_weights_only=True,
    monitor='val_loss', # Salva com base na perda de validação
    mode='min',
    save_best_only=True # Salva apenas o melhor resultado
)

print("\nIniciando o treinamento COMPLETO do modelo...")

# Treinamento final
history = hybrid_model.fit(
    X_train_inputs,
    y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val_inputs, y_val),
    verbose=1,
    callbacks=[model_checkpoint_callback] # Adiciona o callback para salvar
)

print("\nTreinamento completo concluído!")
print(f"Os pesos do melhor modelo foram salvos em '{checkpoint_filepath}'.")

# Opcional: carregar os melhores pesos ao final do treinamento
hybrid_model.load_weights(checkpoint_filepath)

# PASSO 4: Avaliação e Visualização dos Resultados

In [ ]:
# ==============================================================================
# 6. AVALIAÇÃO E VISUALIZAÇÃO DOS RESULTADOS (VERSÃO PARA TESTE)
# ==============================================================================

# ==============================================================================
# NOVO CÓDIGO - AVALIAÇÃO POR REGIÃO
# ==============================================================================
def evaluate_by_region(model, X_val, y_val, df_val, region_col='continent'):
    # Preparar DataFrame de validação
    df_val = df_val.copy()
    df_val['salary_usd'] = y_val.values

    results = []
    for region in df_val[region_col].unique():
        mask = df_val[region_col] == region
        X_region = {
            'input_ids': X_val['input_ids'][mask],
            'attention_mask': X_val['attention_mask'][mask],
            'structured_input': X_val['structured_input'][mask]
        }
        y_region = y_val[mask]

        loss, mae, rmse = model.evaluate(X_region, y_region, verbose=0)
        results.append({
            'region': region,
            'samples': mask.sum(),
            'mae': mae,
            'rmse': rmse,
            'avg_salary': y_region.mean()
        })

    return pd.DataFrame(results)

# Obter o DataFrame original de validação
df_val = X_val.copy()
region_results = evaluate_by_region(hybrid_model, X_val_inputs, y_val, df_val)

print("\nResultados por Região:")
print(region_results.sort_values('mae'))

# Plot adicional
plt.figure(figsize=(10, 6))
sns.barplot(data=region_results, x='region', y='mae')
plt.title('MAE por Região')
plt.ylabel('Erro Médio Absoluto (USD)')
plt.xticks(rotation=45)
plt.show()